In [1]:
#!pip install python-dotenv sqlalchemy psycopg2-binary pandas

In [26]:
import os
import pandas as pd
import numpy as np
import psycopg2 
from psycopg2 import errors
from dotenv import load_dotenv
from sqlalchemy import create_engine



In [4]:
# Cargar las variables de entorno desde .env
load_dotenv()

# Conexión general
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")

# Base de datos origen
dbname = os.getenv("DB_NAME")

# Bodega de datos
dwname = os.getenv("DW_NAME")

print(f"{dbname} {dwname}")


Base_Datos_Proyecto Bodega_Datos_Proyecto


In [5]:
# Engine para base origen
engine_db = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}")

# Engine para la bodega de datos
engine_dw = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dwname}")


In [6]:
query = "SELECT table_name FROM information_schema.tables WHERE table_schema = 'public' AND table_type = 'BASE TABLE';"

df = pd.read_sql(query, engine_dw)

df.head()

,table_name
0,dim_fecha
1,dim_cliente
2,dim_mensajero
3,dim_sede
4,dim_hora


## Dimensión cliente

In [7]:
# 1. Leer las tablas necesarias desde la base de datos original
df_cliente = pd.read_sql("SELECT * FROM cliente;", con=engine_db)
df_ciudad = pd.read_sql("SELECT * FROM ciudad;", con=engine_db)
df_departamento = pd.read_sql("SELECT * FROM departamento;", con=engine_db)

# 2. Renombrar columnas necesarias para estandarizar
df_ciudad = df_ciudad.rename(columns={
    'ciudad_id': 'id',
    'nombre': 'nombre_ciudad',
    'departamento_id': 'departamento_id'
})

df_departamento = df_departamento.rename(columns={
    'departamento_id': 'id',
    'nombre': 'nombre_departamento'
})

df_cliente = df_cliente.rename(columns={
    'nombre': 'nombre_cliente'
})

# 3. Realizar joins con pandas para construir la dimensión cliente
df_dim_cliente = df_cliente \
    .merge(df_ciudad, left_on='ciudad_id', right_on='id', suffixes=('', '_ciudad')) \
    .merge(df_departamento, left_on='departamento_id', right_on='id', suffixes=('', '_depto'))

# 4. Seleccionar columnas deseadas y ordenar
df_dim_cliente = df_dim_cliente[['cliente_id', 'nit_cliente', 'nombre_cliente']]
df_dim_cliente = df_dim_cliente.sort_values(by='cliente_id').reset_index(drop=True)

# 5. Crear la columna llave incremental
df_dim_cliente.insert(0, 'key_dim_cliente', range(len(df_dim_cliente)))

# 6. Guardar la tabla en la bodega de datos
df_dim_cliente.to_sql('dim_cliente', con=engine_dw, if_exists='replace', index=False)

# 7. Verificar
df_dim_cliente.head()



,key_dim_cliente,cliente_id,nit_cliente,nombre_cliente
0,0,1,25,Cliente 2
1,1,2,123,Cliente 1
2,2,3,312289-5,BANCO REGIONAL DE SANGRE BLOD-LIFE
3,3,4,306215-0,CRUZ AZUL-LIFE
4,4,5,300513-3,CLINICA CALI -JOVEN


In [ ]:
# Conectar a la bodega de datos
conn = psycopg2.connect(
    dbname=dwname,
    user=user,
    password=password,
    host=host,
    port=port
)

# Crear cursor
cur = conn.cursor()

try:
    # Intentar añadir la primary key
    cur.execute("""
        ALTER TABLE dim_cliente
        ADD CONSTRAINT pk_dim_cliente PRIMARY KEY (key_dim_cliente);
    """)
    conn.commit()
    print("Primary key creada exitosamente.")

except errors.DuplicateObject:
    print("Ya existe una primary key en la tabla 'dim_cliente'. No se realizó ningún cambio.")

except Exception as e:
    print("Ocurrió un error:", e)

finally:
    # Cerrar cursor y conexión
    cur.close()
    conn.close()

Primary key creada exitosamente.


## Dimensión mensajero

In [9]:
# 1. Leer las tablas necesarias desde la base de datos origen
df_mensajero   = pd.read_sql("SELECT * FROM clientes_mensajeroaquitoy;", con=engine_db)
df_auth_user   = pd.read_sql("SELECT id, username FROM auth_user;", con=engine_db)
df_ciudad      = pd.read_sql("SELECT * FROM ciudad;", con=engine_db)
df_departamento = pd.read_sql("SELECT * FROM departamento;", con=engine_db)

# 2. Renombrar columnas para normalizar nombres
df_ciudad = df_ciudad.rename(columns={
    'ciudad_id': 'id',
    'nombre':    'nombre_ciudad',
    'departamento_id': 'departamento_id'
})

df_departamento = df_departamento.rename(columns={
    'departamento_id': 'id',
    'nombre':          'nombre_departamento'
})

# 3. Construir la dimensión mensajero con merges de pandas
df_dim_mensajero = (
    df_mensajero
    .merge(df_auth_user, left_on='user_id', right_on='id', suffixes=('', '_user'))
    .merge(df_ciudad,  how='left', left_on='ciudad_operacion_id', right_on='id', suffixes=('', '_ciudad'))
    .merge(df_departamento, how='left', left_on='departamento_id', right_on='id', suffixes=('', '_depto'))
)

# 4. Seleccionar y renombrar columnas finales
df_dim_mensajero = df_dim_mensajero[[ 
    'id',
    'username',
    'activo',
    'fecha_entrada',
    'nombre_ciudad',
    'nombre_departamento'
]].rename(columns={
    'id': 'mensajero_id',
    'username': 'nombre',
    'nombre_ciudad': 'ciudad_operacion',
    'nombre_departamento': 'departamento_operacion'
})

# 5. Ordenar por nombre y crear llave incremental
df_dim_mensajero = df_dim_mensajero.sort_values(by='mensajero_id').reset_index(drop=True)
df_dim_mensajero.insert(0, 'key_dim_mensajero', range(len(df_dim_mensajero)))

# 6. Cargar en la bodega de datos
df_dim_mensajero.to_sql('dim_mensajero', con=engine_dw, if_exists='replace', index=False)

# 7. Verificar
df_dim_mensajero.head()


,key_dim_mensajero,mensajero_id,nombre,activo,fecha_entrada,ciudad_operacion,departamento_operacion
0,0,1,mensajero1,True,None,ACOPI YUMBO,VALLE DEL CAUCA
1,1,2,mensajero2,True,None,NaN,NaN
2,2,3,Biil-Gates,True,2012-05-08,CALI,VALLE DEL CAUCA
3,3,4,Lionel_messi,False,2018-12-17,CALI,VALLE DEL CAUCA
4,4,5,James Rodriguez,True,2015-07-01,NaN,NaN


In [ ]:
# Conectar a la bodega de datos
conn = psycopg2.connect(
    dbname=dwname,
    user=user,
    password=password,
    host=host,
    port=port
)

# Crear cursor
cur = conn.cursor()

try:
    # Intentar añadir la primary key
    cur.execute("""
        ALTER TABLE dim_mensajero
        ADD CONSTRAINT pk_dim_mensajero PRIMARY KEY (key_dim_mensajero);
    """)
    conn.commit()
    print("Primary key creada exitosamente.")

except errors.DuplicateObject:
    print("Ya existe una primary key en la tabla 'dim_mensajero'. No se realizó ningún cambio.")

except Exception as e:
    print("Ocurrió un error:", e)

finally:
    # Cerrar cursor y conexión
    cur.close()
    conn.close()

## Dimensión sede

In [11]:
# 1. Leer las tablas necesarias desde la base de datos origen
df_sede        = pd.read_sql("SELECT * FROM sede;", con=engine_db)
df_ciudad      = pd.read_sql("SELECT * FROM ciudad;", con=engine_db)
df_departamento = pd.read_sql("SELECT * FROM departamento;", con=engine_db)

# 2. Renombrar columnas para mantener consistencia
df_ciudad = df_ciudad.rename(columns={
    'ciudad_id': 'id',
    'nombre': 'nombre_ciudad',
    'departamento_id': 'departamento_id'
})

df_departamento = df_departamento.rename(columns={
    'departamento_id': 'id',
    'nombre': 'nombre_departamento'
})

df_sede = df_sede.rename(columns={
    'nombre': 'nombre_sede'
})

# 3. Construir la dimensión sede haciendo los joins necesarios
df_dim_sede = (
    df_sede
    .merge(df_ciudad, left_on='ciudad_id', right_on='id', suffixes=('', '_ciudad'))
    .merge(df_departamento, left_on='departamento_id', right_on='id', suffixes=('', '_depto'))
)

# 4. Seleccionar y renombrar las columnas finales
df_dim_sede = df_dim_sede[[ 
    'sede_id',
    'nombre_sede',
    'direccion',
    'nombre_ciudad',
    'nombre_departamento'
]].rename(columns={
    'nombre_ciudad': 'ciudad_sede',
    'nombre_departamento': 'departamento_sede'
})

# 5. Ordenar por nombre_sede y agregar la columna key_dim_sede
df_dim_sede = df_dim_sede.sort_values(by='sede_id').reset_index(drop=True)
df_dim_sede.insert(0, 'key_dim_sede', range(len(df_dim_sede)))


# 6. Cargar en la bodega de datos
df_dim_sede.to_sql('dim_sede', con=engine_dw, if_exists='replace', index=False)

# 7. Verificar visualizando las primeras filas
df_dim_sede.head()


,key_dim_sede,sede_id,nombre_sede,direccion,ciudad_sede,departamento_sede
0,0,1,Sede principal - Cliente1,Los angeles distrito Latino,CALI,VALLE DEL CAUCA
1,1,2,sede aux - cliente 1,Los angeles distrito Latino,CALI,VALLE DEL CAUCA
2,2,3,TORRES DE MARACAIBO,Los angeles distrito Latino,CALI,VALLE DEL CAUCA
3,3,4,INGENIO,Los angeles distrito Latino,CALI,VALLE DEL CAUCA
4,4,5,VASQUEZ COBO,Los angeles distrito Latino,CALI,VALLE DEL CAUCA


In [22]:
# Conectar a la bodega de datos
conn = psycopg2.connect(
    dbname=dwname,
    user=user,
    password=password,
    host=host,
    port=port
)

# Crear cursor
cur = conn.cursor()

try:
    # Intentar añadir la primary key
    cur.execute("""
        ALTER TABLE dim_sede
        ADD CONSTRAINT pk_dim_sede PRIMARY KEY (key_dim_sede);
    """)
    conn.commit()
    print("Primary key creada exitosamente.")

except errors.DuplicateObject:
    print("Ya existe una primary key en la tabla 'dim_sede'. No se realizó ningún cambio.")

except Exception as e:
    print("Ocurrió un error:", e)

finally:
    # Cerrar cursor y conexión
    cur.close()
    conn.close()


Ocurrió un error: no se permiten múltiples llaves primarias para la tabla «dim_sede»



## Dimensión fecha

In [13]:
# 1. Crear un rango de fechas para 2023 y 2024
fechas = pd.date_range(start='2023-01-01', end='2024-12-31', freq='D')

# 2. Crear dataframe con la columna fecha
df_dim_fecha = pd.DataFrame({'fecha': fechas})

# 3. Extraer componentes: id, dia, mes, nombre_mes, nombre_dia
df_dim_fecha['key_dim_fecha'] = df_dim_fecha.index  # O algún id secuencial
df_dim_fecha['dia'] = df_dim_fecha['fecha'].dt.day
df_dim_fecha['mes'] = df_dim_fecha['fecha'].dt.month
df_dim_fecha['nombre_mes'] = df_dim_fecha['fecha'].dt.month_name(locale='es_ES')  # nombre del mes en español
df_dim_fecha['nombre_dia'] = df_dim_fecha['fecha'].dt.day_name(locale='es_ES')    # nombre del día en español

# 4. Reordenar columnas si quieres
df_dim_fecha = df_dim_fecha[['key_dim_fecha', 'fecha', 'dia', 'mes', 'nombre_mes', 'nombre_dia']]

# 5. Guardar la dimensión en la bodega de datos
df_dim_fecha.to_sql('dim_fecha', con=engine_dw, if_exists='replace', index=False)

# 6. Mostrar resultado
print(df_dim_fecha.head(10))



   key_dim_fecha      fecha  dia  mes nombre_mes nombre_dia
0              0 2023-01-01    1    1      Enero    Domingo
1              1 2023-01-02    2    1      Enero      Lunes
2              2 2023-01-03    3    1      Enero     Martes
3              3 2023-01-04    4    1      Enero  Miércoles
4              4 2023-01-05    5    1      Enero     Jueves
5              5 2023-01-06    6    1      Enero    Viernes
6              6 2023-01-07    7    1      Enero     Sábado
7              7 2023-01-08    8    1      Enero    Domingo
8              8 2023-01-09    9    1      Enero      Lunes
9              9 2023-01-10   10    1      Enero     Martes


In [24]:
# Conectar a la bodega de datos
conn = psycopg2.connect(
    dbname=dwname,
    user=user,
    password=password,
    host=host,
    port=port
)

# Crear cursor
cur = conn.cursor()

try:
    # Intentar añadir la primary key
    cur.execute("""
        ALTER TABLE dim_fecha
        ADD CONSTRAINT pk_dim_fecha PRIMARY KEY (key_dim_fecha);
    """)
    conn.commit()
    print("Primary key creada exitosamente.")

except errors.DuplicateObject:
    print("Ya existe una primary key en la tabla 'dim_fecha'. No se realizó ningún cambio.")

except Exception as e:
    print("Ocurrió un error:", e)

finally:
    # Cerrar cursor y conexión
    cur.close()
    conn.close()

Primary key creada exitosamente.


## Dimensión hora

In [15]:
# Crear un rango de tiempo para todas las horas del día con segundos
# Total segundos en un día = 24*60*60 = 86400
total_seconds = 24 * 60 * 60

# Crear un DataFrame con todas las horas, minutos y segundos del día
df_hora = pd.DataFrame({
    'segundos_del_dia': range(total_seconds)
})

# Convertir segundos del día a tiempo
df_hora['hora_completa'] = pd.to_timedelta(df_hora['segundos_del_dia'], unit='s')

# Extraer hora, minuto y segundo
df_hora['hora'] = df_hora['hora_completa'].dt.components['hours']
df_hora['minuto'] = df_hora['hora_completa'].dt.components['minutes']
df_hora['segundo'] = df_hora['hora_completa'].dt.components['seconds']

# Crear columna con hora en formato HH:MM:SS
df_hora['hora_formateada'] = df_hora['hora_completa'].astype(str).str.slice(7, 15)  # tipo 'hh:mm:ss'

# Asignar un id único (opcional)
df_hora['key_dim_hora'] = df_hora.index

# Reordenar columnas para mejor legibilidad
df_hora = df_hora[['key_dim_hora', 'hora_formateada', 'hora', 'minuto', 'segundo']]

# Guardar en la bodega de datos
df_hora.to_sql('dim_hora', con=engine_dw, if_exists='replace', index=False)

# Mostrar primeras filas
print(df_hora.head())



   key_dim_hora hora_formateada  hora  minuto  segundo
0             0        00:00:00     0       0        0
1             1        00:00:01     0       0        1
2             2        00:00:02     0       0        2
3             3        00:00:03     0       0        3
4             4        00:00:04     0       0        4


In [25]:
# Conectar a la bodega de datos
conn = psycopg2.connect(
    dbname=dwname,
    user=user,
    password=password,
    host=host,
    port=port
)

# Crear cursor
cur = conn.cursor()

try:
    # Intentar añadir la primary key
    cur.execute("""
        ALTER TABLE dim_hora
        ADD CONSTRAINT pk_dim_hora PRIMARY KEY (key_dim_hora);
    """)
    conn.commit()
    print("Primary key creada exitosamente.")

except errors.DuplicateObject:
    print("Ya existe una primary key en la tabla 'dim_hora'. No se realizó ningún cambio.")

except Exception as e:
    print("Ocurrió un error:", e)

finally:
    # Cerrar cursor y conexión
    cur.close()
    conn.close()

Primary key creada exitosamente.


## hecho_mensajería_servicio